In [ ]:
""""""""""""""
#Our model contains the following key layers: cropping ->segmentation -> EfficientNetv2 -> GRU -> output
#For efficient preprocessing, we use seperate code for cropping and segmentation
#This part of the code takes full size (720p or whatever) videos and perform cropping layering work on them to generate 224x224 size videos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# !pip install ultralytics opencv-python-headless
!pip install ultralytics opencv-python-headless

In [ ]:
from ultralytics import YOLO

In [ ]:
striker_model = YOLO('/content/gdrive/MyDrive/Shot Dataset/Other/Saved Training/Detection/FinalPlayerTypeDetection/weights/best.pt') #Player type detection model location which is trained to detect striker
bat_model = YOLO('/content/gdrive/MyDrive/YoloBatDetection/yolov11mpatience20.pt')  #bat detection model location

In [ ]:
import os
import cv2
import numpy as np
from google.colab import files

def process_video(input_video_path, output_video_path, output_size=(224, 224)):
    """
    Process the video to detect and crop the 'Striker' and 'Bat', and save the cropped frames as a video.

    Parameters:
    - input_video_path (str): Path to the input video file.
    - output_video_path (str): Path to the output video file.
    - output_size (tuple): The size to which the cropped frames will be resized.
    """
    # Open the input video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video file {input_video_path}.")
        return

    # Get the frame rate of the input video
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Initialize the video writer for the output video
    fourcc = cv2.VideoWriter_fourcc('M', 'J', 'P', 'G')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, output_size)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Perform detection using both models
        results = striker_model(frame, verbose=False)  # Striker detection
        bat_results = bat_model(frame, verbose=False, max_det=1)  # Bat detection

        # Initialize variables for striker detection
        striker_detected = False
        striker_box = None

        # Find "Striker" bounding box
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                cls = int(box.cls[0].cpu().numpy())
                label = striker_model.names[cls]

                if label == 'Striker':
                    # Striker detected
                    striker_detected = True
                    striker_box = (x1, y1, x2, y2)
                    break
            if striker_detected:
                break

        if striker_detected:
            # Initialize bat detection variables
            bat_detected = False
            bat_box = None

            # Find "Bat" bounding box
            for bat_result in bat_results:
                bat_boxes = bat_result.boxes
                for bat_box_item in bat_boxes:
                    bat_x1, bat_y1, bat_x2, bat_y2 = bat_box_item.xyxy[0].cpu().numpy().astype(int)
                    bat_cls = int(bat_box_item.cls[0].cpu().numpy())
                    bat_label = bat_model.names[bat_cls]

                    if bat_label == 'Cricket Bat':
                        bat_detected = True
                        bat_box = (bat_x1, bat_y1, bat_x2, bat_y2)
                        break
                if bat_detected:
                    break

            # Determine the final cropping coordinates
            if bat_detected:
                # Combine "Striker" and "Bat" bounding boxes
                final_x1 = min(striker_box[0], bat_box[0])
                final_y1 = min(striker_box[1], bat_box[1])
                final_x2 = max(striker_box[2], bat_box[2])
                final_y2 = max(striker_box[3], bat_box[3])
            else:
                # Use "Striker" bounding box only
                final_x1, final_y1, final_x2, final_y2 = striker_box

            # Add 10% padding
            width = final_x2 - final_x1
            padding = int(width * 0.1)
            final_x1_expanded = max(0, final_x1 - padding)
            final_x2_expanded = min(frame.shape[1], final_x2 + padding)

            # Crop the frame with expanded coordinates
            cropped_frame = frame[final_y1:final_y2, final_x1_expanded:final_x2_expanded]

            # Resize the cropped frame
            resized_frame = cv2.resize(cropped_frame, output_size)

            # Write the resized frame to the output video
            out.write(resized_frame)

    # Release resources
    cap.release()
    out.release()
    # cv2.destroyAllWindows()



def load_data(data_dir, output_dir):
    """
    Processes all videos in the given directory and saves the cropped videos to the output directory.

    Parameters:
    - data_dir (str): Path to the main data directory containing subdirectories of videos.
    - output_dir (str): Path to the directory where the output videos will be saved.
    """
    # Loop through all folders in the data directory
    for label in os.listdir(data_dir):
        label_folder = os.path.join(data_dir, label)

        # Ensure that it's a folder (not a file)
        if os.path.isdir(label_folder):
            output_label_dir = os.path.join(output_dir, label)
            os.makedirs(output_label_dir, exist_ok=True)

            # Process all videos in the current folder
            for video in os.listdir(label_folder):
                video_path = os.path.join(label_folder, video)

                # Skip if not a video file (optional, depending on the content of the folder)
                if not video.endswith(('.mp4', '.avi', '.mov')):
                    continue

                video_name, _ = os.path.splitext(video)
                output_video_path = os.path.join(output_label_dir, f"{video_name}.avi")

                print(f"Processing video: {video_path}")
                process_video(video_path, output_video_path)


In [ ]:
"Saves each cropped video in output locations"
#[change paths accordingly]

data_dir = '/content/gdrive/MyDrive/Shot Dataset/Reserved Shots'
output_dir = '/content/gdrive/MyDrive/Shot Dataset/Reserved Shots Crop'
load_data(data_dir, output_dir)